In [0]:
# Setup parameters
dbutils.widgets.text("ClientContainer", "claimsprocessing", "Client Catalog Name")
client_container = dbutils.widgets.get("ClientContainer").strip()
print(f"Executing Alert Group Pipeline for Catalog: {client_container}")

In [0]:
# Step 1: Ensure Target Databases Exist
spark.sql("CREATE DATABASE IF NOT EXISTS claimsprocessing.silver")
spark.sql("CREATE DATABASE IF NOT EXISTS claimsprocessing.gold")

In [0]:
# Step 2: Run Silver & Gold AlertGroup Pipeline
import os, json
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, BooleanType

current_dir = os.getcwd()
if os.path.basename(current_dir).lower() == "dimalertgroup":
    project_root = os.path.abspath(os.path.join(current_dir, ".."))
    config_silver = os.path.join(current_dir, "Gold", "Config", "salertgroup.json")
    config_gold = os.path.join(current_dir, "Gold", "Config", "gdimalertgroup.json")
else:
    project_root = current_dir
    config_silver = os.path.join(current_dir, "dimAlertGroup", "Gold", "Config", "salertgroup.json")
    config_gold = os.path.join(current_dir, "dimAlertGroup", "Gold", "Config", "gdimalertgroup.json")

# 1. Load Raw CSV into temp view AlertGroup for Step 1 Silver
raw_csv_path = os.path.join(project_root, "source", "AlertGroup", "alert_group_reference.csv")
print(f"Loading raw AlertGroup CSV from: {raw_csv_path}")

schema = StructType([
    StructField("AlertGroupID", IntegerType(), False),
    StructField("AlertGroupCode", StringType(), False),
    StructField("AlertGroupDescription", StringType(), False),
    StructField("DisplayText", StringType(), False),
    StructField("SortOrder", IntegerType(), False),
    StructField("Active", BooleanType(), False)
])

df_raw = spark.read.format("csv").option("header", "true").schema(schema).load(raw_csv_path)
df_raw.createOrReplaceTempView("AlertGroup")
print(f"Loaded {df_raw.count()} raw AlertGroup rows.")

# 2. Step 1 Silver Staging (salertgroup.json)
print("=== Step 1: Processing Silver Staging (silver_alertgroup) ===")
with open(config_silver, "r") as f:
    c_silver = json.load(f)["SubLayerProcessing"][0]

df_silver_updates = spark.sql(c_silver["SQLScript"])
df_silver_updates.createOrReplaceTempView("temp_updates")

# Ensure Silver Table Exists
spark.sql("""
CREATE TABLE IF NOT EXISTS claimsprocessing.silver.silver_alertgroup (
 alertGroupID           int
,alertGroupCode         string
,alertGroupDescription  string
,displayText            string
,sortOrder              int
,isActive               boolean
,hashKey                int
) USING delta;
""")

merge_silver = c_silver["MergeScript"].replace("tempSQLScript", "temp_updates")
spark.sql(merge_silver)
print("Silver Staging completed successfully.")

# 3. Step 2 Gold Conformed Dimension (gdimalertgroup.json)
print("=== Step 2: Processing Gold Dimension (gold_dimalertgroup) ===")
with open(config_gold, "r") as f:
    c_gold = json.load(f)["SubLayerProcessing"][0]

# Point view 'alertGroup' to the Silver table created in Step 1!
spark.table("claimsprocessing.silver.silver_alertgroup").createOrReplaceTempView("alertGroup")

df_gold_updates = spark.sql(c_gold["SQLScript"])
df_gold_updates.createOrReplaceTempView("temp_updates")

# Ensure Gold Table Exists
spark.sql("""
CREATE TABLE IF NOT EXISTS claimsprocessing.gold.gold_dimalertgroup (
 alertGroupKey          int
,alertGroupCode         string
,alertGroupDescription  string
,displayText            string
,sortOrder              int
,isActive               boolean
) USING delta;
""")

merge_gold = c_gold["MergeScript"].replace("tempSQLScript", "temp_updates")
spark.sql(merge_gold)
print("=== dimAlertGroup Gold Load completed successfully! ===")

In [0]:
%sql
SELECT * FROM claimsprocessing.gold.gold_dimalertgroup ORDER BY sortOrder;